# Random Forest - Previsão de Churn

**FGV - Modelagem Preditiva**

**Grupo:** João Victor Fortunato Teixeira Mundim, Martinho Aloisio Lenz, Rodrigo Pimentel, Bruno Barradas

Este notebook implementa Random Forest (Ensemble de Árvores de Decisão) para prever Churn de clientes de Internet.

In [ ]:
####################################
# RANDOM FOREST - CHURN            #
####################################

# ===============================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ===============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings("ignore")

print("Bibliotecas importadas com sucesso.")

In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO - Monte o Google Drive ou ajuste os caminhos
# ============================================================

# Para Google Colab, descomente as linhas abaixo:
# from google.colab import drive
# drive.mount('/content/drive')
# CAMINHO_BASE = '/content/drive/MyDrive/aula-predicao/atividade_avaliacao/dados/'

# Para execução local:
CAMINHO_BASE = '../dados/'

ARQUIVO_TREINO = CAMINHO_BASE + 'Churn_de_clientes_de_Internet_-_dados_de_treinamento_8122.xlsx'
ARQUIVO_CLASSIFICAR = CAMINHO_BASE + 'Clientes_reais_para_serem_classificados_c7fc.xlsx'

print(f"Caminho do arquivo de treino: {ARQUIVO_TREINO}")
print(f"Caminho do arquivo para classificar: {ARQUIVO_CLASSIFICAR}")

In [ ]:
# =============================================
# 3. CARREGAMENTO DOS DADOS DE TREINAMENTO
# =============================================
print("=== Carregando dados de treinamento ===")
df_treino = pd.read_excel(ARQUIVO_TREINO)

print(f"Dimensões dos dados: {df_treino.shape[0]} linhas x {df_treino.shape[1]} colunas")
print(f"\nColunas disponíveis:")
for i, col in enumerate(df_treino.columns):
    print(f"  {i+1}. {col} ({df_treino[col].dtype})")

display(df_treino.head())

In [ ]:
# =============================================
# 4. ANÁLISE EXPLORATÓRIA
# =============================================
print("=== Análise Exploratória ===")

print("\nDistribuição da Classe (Churn):")
print(df_treino['Classe'].value_counts())
print(f"\nProporção de Churn: {df_treino['Classe'].mean()*100:.1f}%")

print("\nValores Ausentes:")
missing = df_treino.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    for col in missing_cols.index:
        print(f"  {col}: {missing_cols[col]} valores ausentes ({missing_cols[col]/len(df_treino)*100:.1f}%)")
else:
    print("  Nenhum valor ausente encontrado.")

# Estatísticas descritivas
print("\nEstatísticas Descritivas:")
display(df_treino.describe())

In [ ]:
# =====================================================================
# 5. PRÉ-PROCESSAMENTO
# =====================================================================
print("=== Pré-processamento ===")

# Remover coluna ID (não é feature preditiva)
colunas_remover = ['ID_Cliente']
df_proc = df_treino.drop(columns=colunas_remover, errors='ignore').copy()

# Separar features e target
X = df_proc.drop(columns=['Classe']).copy()
y = df_proc['Classe'].copy()

colunas_features = list(X.columns)
print(f"\nFeatures utilizadas: {colunas_features}")

# Identificar colunas categóricas e numéricas
colunas_cat = X.select_dtypes(include=['object']).columns.tolist()
colunas_num = X.select_dtypes(include=['number']).columns.tolist()

print(f"\nColunas categóricas: {colunas_cat}")
print(f"Colunas numéricas: {colunas_num}")

# Tratamento de valores ausentes
medianas = {}
for col in colunas_num:
    if X[col].isnull().sum() > 0:
        mediana = X[col].median()
        medianas[col] = mediana
        X[col] = X[col].fillna(mediana)
        print(f"  {col}: valores ausentes preenchidos com mediana = {mediana:.2f}")

for col in colunas_cat:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna('Desconhecido')
        print(f"  {col}: valores ausentes preenchidos com 'Desconhecido'")

# Codificação de variáveis categóricas (Label Encoding)
encoders = {}
for col in colunas_cat:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le
    print(f"  {col}: Label Encoding aplicado - classes: {list(le.classes_)}")

print("\nPré-processamento concluído.")

In [ ]:
# ============================================================
# 6. PARÂMETROS DO MODELO
# ============================================================
print("=== Parâmetros do Random Forest ===")

# Número de árvores na floresta
n_estimators = 100

# Critério de divisão: 'gini' ou 'entropy'
criterion = 'entropy'

# Profundidade máxima das árvores (None = sem limite)
max_depth = None

# Número mínimo de amostras para dividir um nó
min_samples_split = 2

# Número mínimo de amostras em uma folha
min_samples_leaf = 1

# Número de features consideradas em cada divisão
max_features = 'sqrt'

# Usar amostragem bootstrap
bootstrap = True

# Proporção para validação
test_size = 0.2

# Semente aleatória
random_state = 42

print(f"\nParâmetros configurados:")
print(f"  n_estimators      = {n_estimators}")
print(f"  criterion         = {criterion}")
print(f"  max_depth         = {max_depth}")
print(f"  min_samples_split = {min_samples_split}")
print(f"  min_samples_leaf  = {min_samples_leaf}")
print(f"  max_features      = {max_features}")
print(f"  bootstrap         = {bootstrap}")
print(f"  test_size         = {test_size}")
print(f"  random_state      = {random_state}")

In [ ]:
# ==========================================
# 7. DIVISÃO TREINO/TESTE E TREINAMENTO
# ==========================================
print("=== Treinando o modelo ===")

# Dividir dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

print(f"\nDados de treino: {len(X_train)} amostras")
print(f"Dados de teste: {len(X_test)} amostras")

# Criar e treinar o modelo
modelo = RandomForestClassifier(
    n_estimators=n_estimators,
    criterion=criterion,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    max_features=max_features,
    bootstrap=bootstrap,
    random_state=random_state,
    n_jobs=-1
)

modelo.fit(X_train, y_train)
print("\nModelo treinado com sucesso!")

In [ ]:
# ==========================================
# 8. AVALIAÇÃO DO MODELO
# ==========================================
print("=== Avaliação do Modelo ===")

# Predições
y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)
y_proba_test = modelo.predict_proba(X_test)[:, 1]

# Métricas
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)
auc_test = roc_auc_score(y_test, y_proba_test)

print(f"\nAcurácia no treino: {acc_train*100:.2f}%")
print(f"Acurácia no teste: {acc_test*100:.2f}%")
print(f"AUC-ROC no teste: {auc_test:.4f}")

# Validação Cruzada
cv_scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
print(f"\nValidação Cruzada (5-fold):")
print(f"  Média: {cv_scores.mean()*100:.2f}%")
print(f"  Desvio Padrão: {cv_scores.std()*100:.2f}%")

print("\nRelatório de Classificação (Teste):")
print(classification_report(y_test, y_pred_test, target_names=['Não Churn (0)', 'Churn (1)']))

In [ ]:
# ==========================================
# 9. VISUALIZAÇÕES
# ==========================================
print("=== Visualizações ===")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Não Churn', 'Churn'],
            yticklabels=['Não Churn', 'Churn'])
axes[0].set_title('Matriz de Confusão - Random Forest')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auc_test:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[1].set_xlabel('Taxa de Falso Positivo')
axes[1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[1].set_title('Curva ROC - Random Forest')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 10. IMPORTÂNCIA DAS VARIÁVEIS
# ==========================================
print("=== Importância das Variáveis ===")

importancias = pd.DataFrame({
    'Feature': colunas_features,
    'Importância': modelo.feature_importances_
}).sort_values('Importância', ascending=False).reset_index(drop=True)

display(importancias)

# Gráfico de importância
fig, ax = plt.subplots(figsize=(10, 6))
top_features = importancias.head(10)
ax.barh(range(len(top_features)), top_features['Importância'], color='darkorange')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importância')
ax.set_title('Top 10 Features Mais Importantes - Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 11. ANÁLISE DE VOTAÇÃO DAS ÁRVORES
# ==========================================
print("=== Análise de Votação das Árvores ===")

# Analisar a votação para o primeiro cliente do conjunto de teste
idx_exemplo = 0
linha = X_test.iloc[[idx_exemplo]]

# Votos individuais de cada árvore
votos = [int(arvore.predict(linha)[0]) for arvore in modelo.estimators_]
contagem = pd.Series(votos).value_counts()

print(f"\nExemplo de votação (cliente {idx_exemplo + 1} do teste):")
print(f"  Votos para Não Churn (0): {contagem.get(0, 0)} árvores")
print(f"  Votos para Churn (1): {contagem.get(1, 0)} árvores")
print(f"  Classe real: {y_test.iloc[idx_exemplo]}")
print(f"  Classe predita: {modelo.predict(linha)[0]}")

In [ ]:
# ==========================================
# 12. TREINAR MODELO FINAL COM TODOS OS DADOS
# ==========================================
print("=== Treinando modelo final com todos os dados ===")

modelo_final = RandomForestClassifier(
    n_estimators=n_estimators,
    criterion=criterion,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    max_features=max_features,
    bootstrap=bootstrap,
    random_state=random_state,
    n_jobs=-1
)

modelo_final.fit(X, y)
print("Modelo final treinado com 100% dos dados.")

In [ ]:
# ===================================================
# 13. CLASSIFICAR CLIENTES REAIS
# ===================================================
print("=== Carregando clientes para classificação ===")

try:
    df_classificar = pd.read_excel(ARQUIVO_CLASSIFICAR)
    print(f"Arquivo carregado: {ARQUIVO_CLASSIFICAR}")
    print(f"Total de clientes para classificar: {len(df_classificar)}")
    
    # Guardar IDs
    ids_clientes = df_classificar['ID_Cliente'].copy() if 'ID_Cliente' in df_classificar.columns else range(1, len(df_classificar)+1)
    
    # Preparar features
    X_novo = df_classificar.drop(columns=['ID_Cliente', 'Classe'], errors='ignore').copy()
    
    # Aplicar mesmo pré-processamento
    for col in colunas_num:
        if col in X_novo.columns:
            if col in medianas:
                X_novo[col] = X_novo[col].fillna(medianas[col])
    
    for col in colunas_cat:
        if col in X_novo.columns:
            X_novo[col] = X_novo[col].fillna('Desconhecido')
            le = encoders[col]
            X_novo[col] = X_novo[col].astype(str).apply(lambda v: v if v in le.classes_ else le.classes_[0])
            X_novo[col] = le.transform(X_novo[col])
    
    # Garantir mesmas colunas
    X_novo = X_novo[colunas_features]
    
    # Predições
    predicoes = modelo_final.predict(X_novo)
    probabilidades = modelo_final.predict_proba(X_novo)
    
    # Resultados
    resultados = pd.DataFrame({
        'ID_Cliente': ids_clientes,
        'Classe_Predita': predicoes,
        'Probabilidade_Nao_Churn': probabilidades[:, 0],
        'Probabilidade_Churn': probabilidades[:, 1]
    })
    
    resultados['Interpretacao'] = resultados['Classe_Predita'].apply(
        lambda x: 'CHURN - Cliente em risco' if x == 1 else 'NÃO CHURN - Cliente retido'
    )
    
    print("\n=== RESULTADOS DA CLASSIFICAÇÃO ===")
    display(resultados)
    
    print(f"\nResumo:")
    print(f"  Clientes classificados como Churn: {(predicoes == 1).sum()}")
    print(f"  Clientes classificados como Não Churn: {(predicoes == 0).sum()}")
    
except FileNotFoundError:
    print(f"Arquivo não encontrado: {ARQUIVO_CLASSIFICAR}")
    print("Pulando etapa de classificação de clientes reais.")

In [ ]:
# ==========================================
# 14. RESUMO FINAL
# ==========================================
print("="*60)
print("RESUMO - RANDOM FOREST")
print("="*60)
print(f"\nModelo: Random Forest Classifier ({criterion.upper()})")
print(f"Dataset: Churn de Clientes de Internet")
print(f"Total de registros de treinamento: {len(df_treino)}")
print(f"\nParâmetros do Modelo:")
print(f"  - N° de árvores: {n_estimators}")
print(f"  - Critério: {criterion}")
print(f"  - Profundidade máxima: {max_depth}")
print(f"  - Max features: {max_features}")
print(f"\nDesempenho:")
print(f"  - Acurácia (treino): {acc_train*100:.2f}%")
print(f"  - Acurácia (teste): {acc_test*100:.2f}%")
print(f"  - AUC-ROC: {auc_test:.4f}")
print(f"  - Validação Cruzada: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")
print(f"\nTop 3 Features Mais Importantes:")
for i, row in importancias.head(3).iterrows():
    print(f"  {i+1}. {row['Feature']}: {row['Importância']:.4f}")
print("="*60)